In [0]:
create table if not exists identifier(:catalog || ".gold.dim_store") as
select * except(file_path,file_name,ingestion_ts) from identifier(:catalog || ".silver.stores_silver")

In [0]:
create table identifier(:catalog || ".gold.dim_product") as
select * except(tags, last_changed_snapshot_date) from identifier(:catalog || ".silver.products_scd")

In [0]:
select * from identifier(:catalog || ".silver.customers_scd")

In [0]:
create table if not exists identifier(:catalog || ".gold.dim_customer") as
with source as (
  select * from identifier(:catalog || ".silver.customers_scd")
)
select
  row_number() over (order by (select null)) as customer_sk,
  *
from source

In [0]:
create table if not exists identifier(:catalog || ".gold.dim_date") as
select
  int(date_format(dt, 'yyyyMMdd')) as date_key,
  dt as calendar_date,
  date_format(dt, 'EEEE') as day_name,
  weekofyear(dt) as iso_week,
  dayofweek(dt) in (1, 7) as is_weekend
from (
  select explode(sequence(
    date '2026-03-01',
    date '2026-03-31',
    interval 1 day
  )) as dt
)

In [0]:
create table identifier(:catalog || ".gold.fact_orders") as
select o.order_id,
date(o.order_ts) as order_date,
  c.customer_sk,
  o.customer_id,
  o.store_id,
  o.order_status,
  o.payment_method,
  sum(oi.quantity) as units,
  sum(oi.gross_amount) as gross_amount,
  sum(oi.discount_amount) as discount_amount,
  sum(oi.net_amount) as net_item_amount,
  o.delivery_fee,
  (sum(oi.net_amount) + o.delivery_fee) AS order_total,
  o.promised_mins,
  o.actual_mins,
  case
        when o.order_status = 'delivered' then
            case when o.actual_mins <= o.promised_mins then true else false end
        else null
    end as is_on_time
from fk_1361.silver.orders_silver o
inner join fk_1361.silver.order_items_silver oi
  on o.order_id = oi.order_id
inner join fk_1361.gold.dim_customer c
on o.customer_id = c.customer_id
group by
    o.order_id,
    o.order_ts,
    c.customer_sk,
    o.customer_id,
    o.store_id,
    o.order_status,
    o.payment_method,
    o.delivery_fee,
    o.promised_mins,
    o.actual_mins